In [1]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as TF  # Das fixiert den NameError
import matplotlib.pyplot as plt

# Device Selection (GPU/CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Training auf GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Training auf CPU (Langsam!)")

def read_pfm(file):
    with open(file, "rb") as f:
        header = f.readline().decode('utf-8').rstrip()
        if header == 'PF':
            color = True
        elif header == 'Pf':
            color = False
        else:
            raise Exception('Keine PFM Datei.')

        dims = f.readline().decode('utf-8').split()
        width = int(dims[0])
        height = int(dims[1])

        scale = float(f.readline().decode('utf-8').rstrip())
        # Negativer Scale bedeutet Little Endian (Standard bei SceneFlow)
        if scale < 0:
            endian = '<'
            scale = -scale
        else:
            endian = '>'

        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)

        data = np.reshape(data, shape)
        data = np.flipud(data) # PFM speichert von unten nach oben
        
        # WICHTIG: Inf-Werte abfangen
        data[data == np.inf] = 0
        
        return data.copy()

    return data

Training auf GPU: NVIDIA GeForce RTX 3080 Ti


In [2]:
class StereoDataset(Dataset):
    def __init__(self, base_dir, mode='train', use_crop=True, use_augmentation=True):
        super().__init__()
        self.mode = mode
        self.use_crop = use_crop and (mode == 'train')
        self.use_augmentation = use_augmentation and (mode == 'train')
        self.to_tensor = transforms.ToTensor()
        self.CROP_SIZE = (320, 640) # Wenn VRAM knapp: (240, 320)
        
        # Pfad-Konstruktion
        disp_root = os.path.join(base_dir, "FlyingThings3D_subset_disparity", "FlyingThings3D_subset", mode, "disparity")
        self.disp_L_dir = os.path.join(disp_root, "left")
        self.disp_R_dir = os.path.join(disp_root, "right")
        
        img_root = os.path.join(base_dir, "FlyingThings3D_subset_image_clean", "FlyingThings3D_subset", mode, "image_clean")
        self.img_L_dir = os.path.join(img_root, "left")
        self.img_R_dir = os.path.join(img_root, "right")
        
        self.filenames = sorted([f for f in os.listdir(self.img_L_dir) if f.endswith('.png')])
        print(f"[{mode.upper()}] Dataset: {len(self.filenames)} Bilder (Grayscale + Resize 640x480)")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        disp_fname = fname.replace('.png', '.pfm')
        
        # 1. BILDER LADEN (Grayscale 'L') & RESIZE
        L_path = os.path.join(self.img_L_dir, fname)
        R_path = os.path.join(self.img_R_dir, fname)
        
        left_img = Image.open(L_path).convert('L')   # <-- WICHTIG: 'L' mode
        right_img = Image.open(R_path).convert('L')
        
        left_img = left_img.resize((640, 480), Image.BILINEAR)
        right_img = right_img.resize((640, 480), Image.BILINEAR)
        
        # 2. DISPARITÄTEN LADEN & SKALIEREN
        # Helper: Load -> Scale Value -> Resize Map -> Clamp
        def load_and_scale_disp(path):
            d = read_pfm(path)
            d = np.abs(d)
            d[~np.isfinite(d)] = 0
            
            # Skalierungsfaktor (Original Breite -> 640)
            orig_w = d.shape[1]
            scale = 640.0 / orig_w
            
            # Wert skalieren
            d = d * scale
            
            # Map Resizen
            d_t = torch.from_numpy(d.copy()).float().unsqueeze(0).unsqueeze(0) # [1, 1, H, W]
            d_t = F.interpolate(d_t, size=(480, 640), mode='nearest')
            
            return torch.clamp(d_t.squeeze(0), 0, 192.0)

        dl_t = load_and_scale_disp(os.path.join(self.disp_L_dir, disp_fname))
        dr_t = load_and_scale_disp(os.path.join(self.disp_R_dir, disp_fname))
        
        # 3. Bilder zu Tensor
        l_t = self.to_tensor(left_img) # Wird [1, 480, 640]
        r_t = self.to_tensor(right_img)
        
        # 4. AUGMENTATION & CROP
        if self.use_crop:
            i, j, h, w = transforms.RandomCrop.get_params(l_t, output_size=self.CROP_SIZE)
            l_t = TF.crop(l_t, i, j, h, w)
            r_t = TF.crop(r_t, i, j, h, w)
            dl_t = TF.crop(dl_t, i, j, h, w)
            dr_t = TF.crop(dr_t, i, j, h, w)
            
            if self.use_augmentation and random.random() > 0.5:
                # Gamma Augmentation (Helligkeit via Gamma ist stabiler)
                gamma = random.uniform(0.8, 1.2)
                l_t = TF.adjust_gamma(l_t, gamma)
                r_t = TF.adjust_gamma(r_t, gamma)

        return l_t, r_t, dl_t, dr_t

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# 1. HAILO-FRIENDLY RESIDUAL BLOCK (Keep as is, excellent for NPU)
# ============================================================================
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# ============================================================================
# 2. NPU-OPTIMIZED REFINEMENT (Replacing the heavy Hourglass)
# ============================================================================
class RefinementHead(nn.Module):
    """
    Hierarchical 2D Refinement. 
    Hailo-8 Status: NATIVE (Only 2D Convs, no bottleneck/memory lag)
    """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )

    def forward(self, low_disp, high_res_feat):
        # Resize low-res disparity to high-res target
        up_disp = F.interpolate(low_disp, size=high_res_feat.shape[2:], mode='bilinear', align_corners=False)
        # Cat disparity with features (e.g. Image or intermediate features)
        x = torch.cat([up_disp, high_res_feat], dim=1)
        # Predict residual correction
        return up_disp + self.conv(x)

# ============================================================================
# 3. COST VOLUME (Kept for Geometry Stability)
# ============================================================================
class CostVolumeStridedConv(nn.Module):
    def __init__(self, in_channels, cv_shifts, num_groups):
        super().__init__()
        self.cv_shifts, self.num_groups, self.in_channels = cv_shifts, num_groups, in_channels
        self.shift_conv = nn.Conv2d(in_channels, in_channels * cv_shifts, (1, cv_shifts), groups=in_channels, bias=False)
        self._init_shift_weights()
        self.shift_conv.weight.requires_grad = False
    def _init_shift_weights(self):
        with torch.no_grad():
            self.shift_conv.weight.zero_()
            for c in range(self.in_channels):
                for d in range(self.cv_shifts):
                    self.shift_conv.weight[c * self.cv_shifts + d, 0, 0, self.cv_shifts - 1 - d] = 1.0
    def forward(self, feat_l, feat_r):
        B, C, H, W = feat_l.shape
        feat_r_shifted = self.shift_conv(F.pad(feat_r, (self.cv_shifts - 1, 0, 0, 0), mode='replicate'))
        feat_r_shifted = feat_r_shifted.view(B, C, self.cv_shifts, H, W)
        diff = torch.abs(feat_l.unsqueeze(2) - feat_r_shifted).view(B, self.num_groups, C // self.num_groups, self.cv_shifts, H, W)
        cost = diff.mean(dim=2)
        return cost.reshape(B, self.num_groups * self.cv_shifts, H, W)

# ============================================================================
# 4. FINAL NPU-OPTIMIZED CORE MODEL
# ============================================================================
class StereoNet_NPU_Core(nn.Module):
    def __init__(self, max_disp=192, num_groups=4):
        super().__init__()
        self.max_disp = max_disp
        self.cv_shifts = (max_disp // 4) + 1 # CV on 1/4 resolution
        
        # 1. Feature Extractor (Shared)
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False), # 1/2
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        self.layer_1_4 = ResBlock(32, 32, stride=2) # 1/4
        
        # 2. Cost Volume & Aggregator (1/4 scale)
        self.cost_volume = CostVolumeStridedConv(32, self.cv_shifts, num_groups)
        self.aggregator = nn.Sequential(
            nn.Conv2d(num_groups * self.cv_shifts, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, self.cv_shifts, 3, padding=1)
        )

        # 3. Refinement Heads (Replacing Hourglass)
        self.refine_v1 = RefinementHead(1 + 32) # Scale 1/2 (Disp + 1/2 Features)
        self.refine_v2 = RefinementHead(1 + 1)  # Scale 1/1 (Disp + Image)

        # 4. OCC-Head
        self.register_buffer("disp_range", torch.arange(0, self.cv_shifts).view(1, -1, 1, 1).float())
        self.occ_head = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), # Input: Gray-Image + Disp-Map
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1) # Output: Occlusion-Logits
        )

    def core(self, left, right, training=True, epoch=0):
        # 1. Feature Extraction
        f_l_1_2 = self.init_conv(left)       # 1/2 Auflösung
        f_l_1_4 = self.layer_1_4(f_l_1_2)    # 1/4 Auflösung
        f_r_1_4 = self.layer_1_4(self.init_conv(right))
        
        # 2. 1/4 Disparity Calculation (Soft-Argmax)
        cv = self.cost_volume(f_l_1_4, f_r_1_4)
        cost_scores = self.aggregator(cv)
        prob = F.softmax(-cost_scores, dim=1)
        disp_4 = torch.sum(prob * self.disp_range, dim=1, keepdim=True)
        
        # 3. Scale up to 1/2 and Refine
        # Wir interpolieren auf die Größe der f_l_1_2 Features
        # Wichtig: * 2.0, weil die Auflösung verdoppelt wurde
        disp_low = F.interpolate(disp_4, size=f_l_1_2.shape[2:], mode='bilinear', align_corners=False) * 2.0
        disp_v1 = self.refine_v1(disp_low, f_l_1_2)
        
        # 4. Scale up to 1/1 and Refine
        # Wir interpolieren auf die Originalgröße (left.shape)
        # Wichtig: * 2.0, weil wir von 1/2 auf 1/1 gehen
        disp_v1_up = F.interpolate(disp_v1, size=left.shape[2:], mode='bilinear', align_corners=False) * 2.0
        disp_final = self.refine_v2(disp_v1_up, left)
        
        # 5. Alles für den Loss auf volle Auflösung bringen
        # Falls disp_low noch auf 1/2 ist, hier für den Loss hochziehen
        disp_low_up = F.interpolate(disp_low, size=left.shape[2:], mode='bilinear', align_corners=False) * 2.0
        
        # Occlusion Dummy (muss gleiche Größe wie disp_final haben)
        # 1. Normalisiere die Inputs für den Head (hilft bei der GradNorm)
        norm_img = left # ist bereits [0, 1]
        norm_disp = disp_final / self.max_disp

        # 2. Head berechnen
        occ_in = torch.cat([norm_img, norm_disp], dim=1)
        occ = self.occ_head(occ_in)
        
        # WICHTIG: Da disp_range auf 1/4 Skala basiert (0-48), 
        # haben wir durch die zwei *2.0 Schritte oben bereits die Zielskala (0-192) erreicht!
        # Wir geben die Tensoren jetzt alle in 1/1 Auflösung zurück.
        return disp_final, disp_v1_up, disp_low_up, occ

    def forward(self, left, right, training=False, epoch=0):
        return self.core(left, right, training=training, epoch=epoch)

# ============================================================================
# 5. ALL-MODE WRAPPER (LR + RL Pass)
# ============================================================================
class StereoFusionAllMode(nn.Module):
    def __init__(self, max_disp=192):
        super().__init__()
        self.core = StereoNet_NPU_Core(max_disp=max_disp)

    def forward(self, left, right, mode="fast", epoch=0):
        training_needed = (mode != "fast")
        out_LR = self.core(left, right, training=training_needed, epoch=epoch)
        if mode == "fast": return {"LR": out_LR}
        
        # RL Pass (Flip Symmetry)
        out_RL_f = self.core(torch.flip(right, [3]), torch.flip(left, [3]), training=training_needed, epoch=epoch)
        out_RL = tuple(torch.flip(t, [3]) for t in out_RL_f)
        return {"LR": out_LR, "RL": out_RL}

In [4]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    total_epe = 0.0
    total_loss = 0.0
    valid_batches = 0
    
    # tqdm für Fortschrittsbalken
    pbar = tqdm(val_loader, desc="🔍 Validierung", leave=False, ncols=150)
    
    # FIX: Jetzt 4 Werte entpacken statt 3
    for left, right, gt_L, gt_R in pbar:
        left, right = left.to(device), right.to(device)
        gt_L = gt_L.to(device)
        # gt_R brauchen wir für EPE-Validierung eigentlich nicht zwingend, 
        # aber wir müssen es entpacken, damit Python nicht meckert.

        # Forward Pass (Wir nutzen nur LR Core für Speed)
        # model.core gibt zurück: (disp_final, disp_v1, disp_low, occ)
        out = model.core(left, right)
        disp_pred = out[0] # Wir nehmen nur die finale Disparität
        
        # Validitäts-Maske (Nur Pixel prüfen, die Ground Truth haben)
        mask = (gt_L > 0) & (gt_L < 192)
        
        if mask.sum() > 0:
            # 1. EPE (End Point Error) berechnen
            # Absoluter Abstand in Pixeln
            diff = torch.abs(disp_pred[mask] - gt_L[mask])
            epe = diff.mean().item()
            
            # 2. Loss berechnen (Smooth L1 als Referenz)
            loss = F.smooth_l1_loss(disp_pred[mask], gt_L[mask], beta=1.0).item()
            
            total_epe += epe
            total_loss += loss
            valid_batches += 1
            
            pbar.set_postfix({'val_epe': f"{epe:.2f}"})

    if valid_batches == 0:
        return 0.0, 0.0

    return (total_loss / valid_batches), (total_epe / valid_batches)


In [5]:
import torch.nn.functional as F

import torch.nn.functional as F




def edge_aware_smoothness_loss(pred_disp, left_img, beta=15.0):
    """
    Beta=15 schützt Kanten radikal. Nur in sehr flachen Bereichen (Gefälle < 1/15)
    wird geglättet. Das verhindert das 'Verwaschen' deiner Stuhlbeine.
    """
    def grad_x(img): return torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:])
    def grad_y(img): return torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :])

    disp_grad_x = grad_x(pred_disp)
    disp_grad_y = grad_y(pred_disp)

    # Graustufen-Gradienten des Referenzbildes
    img_grad_x = grad_x(left_img)
    img_grad_y = grad_y(left_img)

    weight_x = torch.exp(-img_grad_x * beta)
    weight_y = torch.exp(-img_grad_y * beta)

    return (disp_grad_x * weight_x).mean() + (disp_grad_y * weight_y).mean()




def robust_stereo_loss(outputs, left_img, right_img, gt_disp_L, gt_disp_R, 
                       w_geom=1.0, w_lrc=1.0, w_photo=1.0, w_smooth=0.1):
    """
    Update V8.1: 
    - Inklusive Deep Supervision (3 Level)
    - Inklusive OCC-Head Training (BCE Loss)
    """
    # 0. Vorbereitungen
    disp_pred_L = outputs["LR"][0]  # Finaler Output für Geometrie-Check
    mask_valid_L = (gt_disp_L > 0) & (gt_disp_L < 192)
    B, _, H, W = disp_pred_L.shape

    # --- 1. DEEP SUPERVISION GEOMETRY LOSS ---
    # Wir bestrafen alle 3 Stufen: [final, v1, low]
    # Gewichtung: Final (1.0), V1 (0.5), Low (0.25)
    loss_geom = 0.0
    geom_weights = [1.0, 0.5, 0.25]
    
    for i, weight in enumerate(geom_weights):
        pred_L = outputs["LR"][i]
        loss_geom += weight * F.smooth_l1_loss(pred_L[mask_valid_L], gt_disp_L[mask_valid_L], beta=1.0)
        
        if "RL" in outputs:
            pred_R = outputs["RL"][i]
            mask_valid_R = (gt_disp_R > 0) & (gt_disp_R < 192)
            loss_geom += weight * F.smooth_l1_loss(pred_R[mask_valid_R], gt_disp_R[mask_valid_R], beta=1.0)
    
    # Mittelwert bilden (pro Stufe / pro Auge)
    divisor = len(geom_weights) * (2.0 if "RL" in outputs else 1.0)
    loss_geom /= divisor

    # --- 2. GEOMETRISCHE SICHTBARKEIT (OCCLUSION MASK) ---
    # (Dieser Teil bleibt die Basis für Photo- und LRC-Loss)
    grid_x = torch.arange(W, device=disp_pred_L.device).view(1, 1, 1, W).expand(B, 1, H, W).float()
    x_projected = grid_x - disp_pred_L
    mask_oob = (x_projected >= 0) & (x_projected < W)
    grid_y = torch.arange(H, device=disp_pred_L.device).view(1, 1, H, 1).expand(B, 1, H, W).float()
    norm_x = 2.0 * x_projected / (W - 1) - 1.0
    norm_y = 2.0 * grid_y / (H - 1) - 1.0
    grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
    
    gt_dist_at_target = F.grid_sample(gt_disp_R, grid, align_corners=False, padding_mode='border')
    mask_consist = torch.abs(disp_pred_L - gt_dist_at_target) < 1.5 # Etwas Toleranz für Resize
    mask_vis = (mask_oob & mask_consist).detach().float()

    # --- 3. NEU: OCC-HEAD TRAINING ---
    # Wir trainieren den occ_head (Index 3 im Output), die geometrische Maske vorherzusagen
    occ_logits = outputs["LR"][3]
    # Target: 1.0 wenn verdeckt (mask_vis=0), 0.0 wenn sichtbar (mask_vis=1)
    occ_target = 1.0 - mask_vis
    loss_occ = F.binary_cross_entropy_with_logits(occ_logits, occ_target)

    # --- 4. PHOTO- & LRC-LOSS (Nur sichtbare Pixel) ---
    left_img_warped = F.grid_sample(right_img, grid, align_corners=False, padding_mode='border')
    loss_photo = (torch.abs(left_img - left_img_warped) * mask_vis).sum() / (mask_vis.sum() + 1e-6)

    loss_lrc = torch.tensor(0.0, device=disp_pred_L.device)
    if "RL" in outputs:
        disp_R_warped = F.grid_sample(outputs["RL"][0], grid, align_corners=False, padding_mode='border')
        loss_lrc = (torch.abs(disp_pred_L - disp_R_warped) * mask_vis).sum() / (mask_vis.sum() + 1e-6)

    # --- 5. SMOOTHNESS ---
    loss_smooth = edge_aware_smoothness_loss(disp_pred_L, left_img)

    # Gesamtloss (BCE-Occ mit Gewicht 0.5 einfließen lassen)
    total_loss = (w_geom * loss_geom) + \
                 (w_photo * loss_photo) + \
                 (w_smooth * loss_smooth) + \
                 (w_lrc * loss_lrc) + \
                 (0.5 * loss_occ)
    
    logs = {
        "geom": loss_geom.item(), 
        "photo": loss_photo.item(), 
        "lrc": loss_lrc.item(), 
        "occ": loss_occ.item()
    }
    return total_loss, logs



import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import os
import time

def train_strategic(
    model, 
    base_dir, 
    # --- Training Hyperparameter ---
    epochs=75, 
    lr_max=2e-4, 
    weight_decay=1e-5,
    warmup_pct=0.1,      # Prozent der Epochen für Warmup
    grad_clip=1.0,
    
    # --- Dataloader & Dataset ---
    batch_size=6, 
    num_workers=4, 
    use_crop=True,       # Training mit Random Crop?
    use_aug=True,        # Training mit Color Augmentation?
    
    # --- Strategie-Steuerung ---
    geom_only_epochs=30  # Wie lange Phase 1 (Geometrie) dauert
):
    
    # 1. Datasets & Loader (Voll parametrisiert)
    train_ds = StereoDataset(base_dir, mode='train', use_crop=use_crop, use_augmentation=use_aug)
    val_ds = StereoDataset(base_dir, mode='val', use_crop=False, use_augmentation=False)
    
    train_loader = DataLoader(
        train_ds, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True, 
        persistent_workers=True,
        prefetch_factor=2
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    
    # 2. Optimizer & Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=lr_max, weight_decay=weight_decay)
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr_max, 
        total_steps=epochs * len(train_loader),
        pct_start=warmup_pct, 
        div_factor=25, 
        final_div_factor=1000
    )
    
    scaler = torch.cuda.amp.GradScaler()
    current_best_epe = float('inf')
    print(f"🚀 TRAINING START | {epochs} Epochen | BS={batch_size} | LR={lr_max:.1e}")
    print(f"   Strategie: {geom_only_epochs} Epochen Geometrie -> Dann Refinement")
    
    log_file = "training_log_robust.txt"
    # Header Zeile:
    with open(log_file, "a") as f:
        f.write("epoch\tavg_loss\tval_loss\tval_epe\tgeom\tphoto\tlrc\tlr\tphase\tw_geom\tw_aux\n")

    for epoch in range(epochs):
        model.train()
        # Wir initialisieren die Zähler auf 0
        epoch_loss = 0.0
        sum_logs = {"geom": 0.0, "photo": 0.0, "lrc": 0.0, "occ": 0.0} # 'occ' hinzugefügt

        # --- PHASE MANAGEMENT ---
        if epoch < geom_only_epochs:
            phase = "INITIAL"
            w_geom = 1.0
            w_aux = 0.1 
        else:
            phase = "REFINE"
            # Aux hochfahren (0.1 -> 1.0), Geometrie senken (1.0 -> 0.5)
            progress = (epoch - geom_only_epochs) / (epochs - geom_only_epochs)
            w_geom = 1.0 - 0.5 * progress 
            w_aux = 0.1 + 0.9 * progress   
        
        
            
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1} [{phase}]", ncols=160)
        
        for batch_idx, (l, r, dl, dr) in enumerate(pbar):
            l, r, dl, dr = l.to(device), r.to(device), dl.to(device), dr.to(device)
            
            with torch.cuda.amp.autocast():
                # LR Pass
                out_LR = model.core(l, r)
                outputs = {"LR": out_LR}
                
                # RL Pass (immer aktiv für Konsistenz)
                l_flip, r_flip = torch.flip(l, [3]), torch.flip(r, [3])
                out_RL_flip = model.core(r_flip, l_flip)
                outputs["RL"] = tuple(torch.flip(o, [3]) for o in out_RL_flip)
                
                loss, logs = robust_stereo_loss(
                    outputs, l, r, dl, dr,
                    w_geom=w_geom, w_photo=w_aux, w_lrc=w_aux, w_smooth=w_aux*0.1
                )
                # Hier summieren wir die Werte auf
                epoch_loss += loss.item()
                for k, v in logs.items():
                    if k in sum_logs:
                        sum_logs[k] += v
            
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip).item()
            
            # Damp Berechnung
            curr_lr = scheduler.get_last_lr()[0]
            damp = (curr_lr / (grad_norm + 1e-8)) * 1000
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            vram_gb = torch.cuda.memory_reserved(device) / 1024**3
            pbar.set_postfix({
                'VRAM': f"{vram_gb:.1f}G",
                'Loss': f"{loss.item():.2f}",
                'G': f"{logs['geom']:.2f}",
                'A': f"{logs['photo']:.2f}",
                'Grad': f"{grad_norm:.1f}",
                'Damp': f"{damp:.1f}",
                'LR': f"{curr_lr:.1e}"
            })

        # Validation Step
        val_loss, val_epe = validate(model, val_loader, device)
        

        # --- LOGGING ---
        n_batches = len(train_loader)
        avg_loss = epoch_loss / n_batches
        avg_logs = {k: v / n_batches for k, v in sum_logs.items()}
        
        print(f"📈 Epoche {epoch+1} [{phase}]: Val-EPE: {val_epe:.2f} px")
        
        
        # ... (nach avg_loss Berechnung) ...

        with open(log_file, "a") as f:
            f.write(f"{epoch+1}\t{avg_loss:.4f}\t{val_loss:.4f}\t{val_epe:.4f}\t"
                    f"{avg_logs.get('geom', 0):.4f}\t"    # Ersetzt data_LR
                    f"{avg_logs.get('photo', 0):.4f}\t"   # Ersetzt photo_L
                    f"{avg_logs.get('lrc', 0):.4f}\t"     # Bleibt gleich
                    f"{curr_lr:.2e}\t"
                    f"{phase}\t"
                    f"{w_geom:.2f}\t"                     # Neu: Gewichtung Geometrie
                    f"{w_aux:.2f}\n")                     # Ersetzt warp_factor
        # Visuals
        save_debug_visuals(model, val_loader.dataset, epoch+1, index=6)
        plot_disparity_profile(model, val_loader.dataset, epoch+1, index=6)
        save_preview(model, val_loader.dataset, f"epoch_{(epoch+1):03d}", index=6)
        save_symmetry_comparison(model, val_loader.dataset, device, epoch=epoch+1, index=6)

        # Save Checkpoints
        if val_epe < current_best_epe:
            current_best_epe = val_epe
            torch.save(model.state_dict(), "StereoFusion_BEST.pth")
        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f"StereoFusion_ep_{epoch+1}.pth")
            
    print("✅ Training beendet.") 

     




/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def save_debug_visuals(model, dataset, epoch, index=6):
    model.eval()
    
    # 1. Daten holen & Vorbereiten
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Core liefert 4 Werte: Final, V1, Low, Occ-Logits
        disp_final, disp_v1, disp_low, _ = model.core(l_in, r_in)
        
    # Hilfsfunktion zum sauberen Konvertieren für Matplotlib (Grayscale-Safe)
    def to_img_np(t):
        arr = t.squeeze().cpu().numpy()
        return arr # Bei Grayscale einfach [H, W] zurückgeben

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Debug Visuals - Epoche {epoch}", fontsize=16)

    # Zeile 1: Input & GT & Final Prediction
    axs[0,0].imshow(to_img_np(l), cmap='gray')
    axs[0,0].set_title("Input Image (Left)")
    
    axs[0,1].imshow(to_img_np(dl), cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("Ground Truth Disparity")
    
    im_final = axs[0,2].imshow(to_img_np(disp_final), cmap='magma', vmin=0, vmax=192)
    axs[0,2].set_title("Final Prediction (1/1)")
    fig.colorbar(im_final, ax=axs[0,2])

    # Zeile 2: Die Pyramiden-Stufen
    axs[1,0].imshow(to_img_np(disp_low), cmap='magma', vmin=0, vmax=192)
    axs[1,0].set_title("Stage Low (1/4 Scale)")
    
    axs[1,1].imshow(to_img_np(disp_v1), cmap='magma', vmin=0, vmax=192)
    axs[1,1].set_title("Stage V1 (1/2 Scale)")
    
    # EPE Map (Fehlerbild)
    gt_mask = (dl > 0) & (dl < 192)
    epe_map = np.abs(to_img_np(dl) - to_img_np(disp_final))
    epe_map[~gt_mask.squeeze().numpy()] = 0 
    im_epe = axs[1,2].imshow(epe_map, cmap='jet', vmin=0, vmax=10)
    axs[1,2].set_title("EPE Map (Error)")
    fig.colorbar(im_epe, ax=axs[1,2])

    for ax in axs.flatten(): ax.axis('off')
    plt.tight_layout()
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/debug_ep_{epoch:03d}.png")
    plt.close()

    
import matplotlib.pyplot as plt

def save_preview(model, dataset, name, index=6):
    model.eval()
    
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # A) LR Pass
        disp_LR, _, _, occ_logits = model.core(l_in, r_in)
        
        # B) RL Pass (für Symmetrie-Check / Echo)
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(disp_RL_f, [3])
        
        # C) Echo-Map Grid (LRC Check)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        echo_map = torch.abs(disp_LR - disp_RL_warped)
        
        # D) Okklusions-Wahrscheinlichkeit (Sigmoid)
        occ_prob = torch.sigmoid(occ_logits)

    # Hilfsfunktion für NP-Konvertierung (Grayscale)
    def to_np(t): return t.squeeze().cpu().numpy()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Preview Analysis: {name} (Robot Logic)", fontsize=16)

    # 1. Input
    axes[0, 0].imshow(to_np(l), cmap='gray')
    axes[0, 0].set_title("Input Left")
    
    # 2. GT
    axes[0, 1].imshow(to_np(dl), cmap='magma', vmin=0, vmax=192)
    axes[0, 1].set_title("Ground Truth")
    
    # 3. Vorhersage Disparität
    im_lr = axes[0, 2].imshow(to_np(disp_LR), cmap='magma', vmin=0, vmax=192)
    axes[0, 2].set_title("Predicted Disparity")
    fig.colorbar(im_lr, ax=axes[0, 2])

    # 4. Echo-Map (Wo widersprechen sich LR und RL?)
    im_echo = axes[1, 0].imshow(to_np(echo_map), cmap='hot', vmin=0, vmax=5)
    axes[1, 0].set_title("Echo-Map (LRC Error)")
    fig.colorbar(im_echo, ax=axes[1, 0])

    # 5. NEU: Predicted Occlusion (Was der Roboter "sieht")
    # Weiß = Verdeckt/Ungültig, Schwarz = Sicher/Sichtbar
    im_occ = axes[1, 1].imshow(to_np(occ_prob), cmap='gray', vmin=0, vmax=1)
    axes[1, 1].set_title("Predicted Occlusion Head")
    fig.colorbar(im_occ, ax=axes[1, 1])
    
    # 6. EPE Map
    gt_mask = (dl > 0) & (dl < 192)
    epe_map = np.abs(to_np(dl) - to_np(disp_LR))
    epe_map[~gt_mask.squeeze().numpy()] = 0 
    im_epe = axes[1, 2].imshow(epe_map, cmap='jet', vmin=0, vmax=10)
    axes[1, 2].set_title("EPE Error Map")
    fig.colorbar(im_epe, ax=axes[1, 2])
    
    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout()
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/preview_{name}.png", bbox_inches='tight')
    plt.close(fig)


def plot_disparity_profile(model, dataset, epoch, index=0):
    model.eval()
    
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Unpacking von 4 Werten
        disp_final, disp_v1, disp_low, _ = model.core(l_in, r_in)
        
    # Helper: Squeezed auf [H, W]
    def to_np(t): return t.squeeze().cpu().numpy()
    
    # FIX: Nur transponieren, wenn es 3 Kanäle hat. Bei [H, W] direkt nutzen.
    left_img = to_np(l)
    if left_img.ndim == 3: # Falls [3, H, W]
        left_img = left_img.transpose(1, 2, 0)
    
    gt_np = to_np(dl)
    final_np = to_np(disp_final)
    v1_np = to_np(disp_v1)
    
    H, W = final_np.shape
    rows = [int(H * 0.25), int(H * 0.50), int(H * 0.75)]
    labels = ["25%", "50%", "75%"]
    
    fig, axs = plt.subplots(4, 1, figsize=(12, 16))
    
    # Bild mit Linien (cmap='gray' hinzugefügt)
    axs[0].imshow(left_img, cmap='gray' if left_img.ndim == 2 else None)
    for row in rows: axs[0].axhline(row, color='yellow', linewidth=2)
    axs[0].set_title(f"Profile Lines (Epoch {epoch})")
    axs[0].axis('off')
    
    # Profile
    for i, (row, label) in enumerate(zip(rows, labels)):
        ax = axs[i+1]
        ax.plot(gt_np[row, :], 'k-', label='Ground Truth', linewidth=2)
        ax.plot(final_np[row, :], 'r-', label='Final Prediction', alpha=0.8)
        ax.plot(v1_np[row, :], 'g--', label='V1 Coarse', alpha=0.6)
        
        ax.set_title(f"Profile at {label} Height")
        ax.set_ylim(-5, 200)
        ax.grid(True, alpha=0.3)
        if i==0: ax.legend()
        
    plt.tight_layout()
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/profile_ep{epoch:03d}.png")
    plt.close(fig)

def save_symmetry_comparison(model, dataset, device, epoch=0, index=0):
    model.eval()
    l, r, dl, dr = dataset[index]
    l_in, r_in = l.unsqueeze(0).to(device), r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # FIX 1: Variable hieß vorher disp_RL_f, muss aber disp_LR sein
        # FIX 2: Unpacking von 4 Werten
        disp_LR, _, _, _ = model.core(l_in, r_in)
        
        # RL Pass (Inputs flippen)
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        # FIX 3: Auch hier 4 Werte entpacken
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(disp_RL_f, [3]) 
        
        # LRC Check (Warp RL to Left)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        # FIX 4: squeeze(1) vor stack (wie im Loss Fix)
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        lrc_diff = torch.abs(disp_LR - disp_RL_warped)

    # Plotting Helper
    def to_np(t): return t.squeeze().cpu().numpy()
    
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    # 1. LR Prediction
    im1 = axs[0,0].imshow(to_np(disp_LR), cmap='magma', vmin=0, vmax=192)
    axs[0,0].set_title("LR Prediction")
    fig.colorbar(im1, ax=axs[0,0])
    
    # 2. RL Prediction (Warped to Left)
    im2 = axs[0,1].imshow(to_np(disp_RL_warped), cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("RL Prediction (Warped to Left)")
    
    # 3. LRC Differenz
    im3 = axs[1,0].imshow(to_np(lrc_diff), cmap='hot', vmin=0, vmax=10)
    axs[1,0].set_title("LRC Difference (Consistency Check)")
    fig.colorbar(im3, ax=axs[1,0])
    
    # 4. Bild Overlay (Fix für Grayscale)
    left_img = to_np(l)
    if left_img.ndim == 3:
        left_img = left_img.transpose(1, 2, 0)
    axs[1,1].imshow(left_img, cmap='gray' if left_img.ndim == 2 else None)
    axs[1,1].set_title("Left Image")

    for ax in axs.flat: ax.axis('off')
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/symmetry_ep{epoch:03d}.png")
    plt.close(fig)


In [ ]:
def main():
    torch.cuda.empty_cache()
    global device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Modell Setup
    model = StereoFusionAllMode(max_disp=192).to(device)
    # BatchNorm Momentum Reset
    model.apply(lambda m: setattr(m, 'momentum', 0.01) if isinstance(m, torch.nn.BatchNorm2d) else None)

    # --- KONFIGURATION ---
    train_strategic(
        model=model,
        base_dir=r"/home/slarc/datasets/sceneflow",  # Dein Root Pfad
        
        # Hyperparams
        epochs=75,
        lr_max=2e-4,          # Maximaler LR Peak
        weight_decay=1e-5,
        warmup_pct=0.1,       # Erste 10% (7 Epochen) Warmup
        grad_clip=1.0,        # Stabilisiert Training
        
        # Loader & Data
        batch_size=16,         # Ggf. auf 4 senken bei OOM
        num_workers=4,
        use_crop=True,        # WICHTIG: True für Training (Random Crops)
        use_aug=False,         # WICHTIG: True für Robustheit
        
        # Strategie
        geom_only_epochs=30   # Nach 30 Epochen schalten wir Aux dazu
    )

if __name__ == '__main__':
    main()




/tmp/ipykernel_3689937/3716136832.py:174: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[TRAIN] Dataset: 21818 Bilder (Grayscale + Resize 640x480)
[VAL] Dataset: 4248 Bilder (Grayscale + Resize 640x480)
🚀 TRAINING START | 75 Epochen | BS=16 | LR=2.0e-04
   Strategie: 30 Epochen Geometrie -> Dann Refinement


Ep 1 [INITIAL]:   0%|                                                                                                                  | 0/1364 [00:00<?, ?it/s]/tmp/ipykernel_3689937/3716136832.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Ep 1 [INITIAL]: 100%|████████████████████████████████| 1364/1364 [07:34<00:00,  3.00it/s, VRAM=8.3G, Loss=5.85, G=5.29, A=0.02, Grad=

📈 Epoche 1 [INITIAL]: Val-EPE: 11.56 px


Ep 2 [INITIAL]: 100%|████████████████████████████████| 1364/1364 [07:31<00:00,  3.02it/s, VRAM=8.3G, Loss=7.21, G=6.72, A=0.02, Grad=15.8, Damp=0.0, LR=4.0e-05]
                                                                                                                                                      

...

Ep 50 [REFINE]: 100%|████████████████████████████████| 1364/1364 [07:40<00:00,  2.96it/s, VRAM=8.3G, Loss=1.37, G=1.32, A=0.01, Grad=25.7, Damp=0.0, LR=6.0e-05]
                                                                                                                                                      

📈 Epoche 50 [REFINE]: Val-EPE: 2.14 px


Ep 51 [REFINE]: 100%|████████████████████████████████| 1364/1364 [07:33<00:00,  3.01it/s, VRAM=8.3G, Loss=1.66, G=1.62, A=0.01, Grad=16.0, Damp=0.0, LR=5.6e-05]
                                                                                                                                                      

📈 Epoche 51 [REFINE]: Val-EPE: 2.21 px


Ep 52 [REFINE]:   3%|█                                 | 44/1364 [00:15<06:54,  3.19it/s, VRAM=8.3G, Loss=1.04, G=0.92, A=0.01, Grad=10.3, Damp=0.0, LR=5.6e-05]

In [ ]:
%abort

In [ ]:
import matplotlib.pyplot as plt
full_dataset = StereoDataset(
        left_dir='/home/slarc/datasets/sceneflow/left',
        right_dir='/home/slarc/datasets/sceneflow/right',
        disp_dir='/home/slarc/datasets/sceneflow/disp',
        training=True
    )
val_ratio = 0.1
val_size = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

# Ein Bild aus dem Dataset holen
left, right, gt = train_dataset[7491] 

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(left.squeeze(), cmap='gray')
plt.title("Eingangsbild (Ist es aufrecht?)")

plt.subplot(1, 3, 2)
plt.imshow(gt.squeeze(), cmap='jet')
plt.title("GT Disparität")

# Check: Wo sind die Gradienten am stärksten?
# Wenn dy > dx, dann ist die Disparität vertikal orientiert!
dy, dx = torch.gradient(gt.squeeze())
plt.subplot(1, 3, 3)
plt.imshow(dx.abs() > dy.abs(), cmap='gray')
plt.title("Weiß = Horizontale Struktur\nSchwarz = Vertikale Struktur")
plt.show()

In [ ]:
# Testen Sie:
feat = make_feature_extractor()
dummy = torch.randn(1, 1, 480, 640)
out = feat(dummy)
print(f"Feature channels: {out.shape[1]}")  # Muss 32 sein!

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from PIL import Image

# ---------------------------------------------------------
# Hilfsfunktion: Bild laden und normalisieren
# ---------------------------------------------------------
def load_gray_image(path):
    img = Image.open(path).convert("L")
    t = transforms.ToTensor()
    return t(img).unsqueeze(0).cuda()

# Pfade und Device
left_path  = "/home/slarc/datasets/sceneflow/left/0000006.png"
right_path = "/home/slarc/datasets/sceneflow/right/0000006.png"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

left  = load_gray_image(left_path)
right = load_gray_image(right_path)

# Modell laden (Stelle sicher, dass die Klasse definiert ist)
model = StereoFusionAllMode().to(device)
model.eval()

with torch.no_grad():
    # 1. Fast Mode (Nur LR Pass)
    out_fast = model(left, right, mode="fast")
    # Extraktion aus dem Dictionary-Key "LR"
    disp_fast = out_fast["LR"][0]
    occ_fast  = out_fast["LR"][3]

    # 2. Precise Mode (LR und geflippter RL Pass)
    out_prec = model(left, right, mode="precise")
    disp_prec_LR = out_prec["LR"][0]   # Normaler Pass
    disp_prec_RL = out_prec["RL"][0]   # Symmetrischer RL-Pass (bereits zurückgeflippt!)
    
    # Echo-Analyse: Wo unterscheiden sich LR und RL? (Meist am linken Rand)
    echo_map = torch.abs(disp_prec_LR - disp_prec_RL)

# ---------------------------------------------------------
# Visualisierung: Der "Miststück-Check"
# ---------------------------------------------------------
def show_disp(disp, title, subplot_pos, cmap="magma"):
    plt.subplot(2, 2, subplot_pos)
    disp_np = disp.squeeze().cpu().numpy()
    plt.imshow(disp_np, cmap=cmap, vmin=0, vmax=192)
    plt.colorbar(label="Pixel")
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(16, 10))

# Oben Links: Fast Mode (Standard)
show_disp(disp_fast, "Fast Mode (LR only)", 1)

# Oben Rechts: Precise Mode LR
show_disp(disp_prec_LR, "Precise Mode (LR Pass)", 2)

# Unten Links: Precise Mode RL (Der Retter für den linken Rand)
show_disp(disp_prec_RL, "Precise Mode (RL Pass - Flipped)", 3)

# Unten Rechts: Echo-Analyse (LRC-Diff)
# Hier siehst du die Fehler am linken Rand leuchten!
show_disp(echo_map, "Echo Analysis (LRC Diff)", 4, cmap="hot")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# EPE Check
# ---------------------------------------------------------
# Falls gt_disp.npy nicht existiert, erstellen wir eine Dummy-Maske zum Testen
try:
    gt = np.load("gt_disp.npy")
    gt = torch.tensor(gt, dtype=torch.float32).unsqueeze(0).unsqueeze(0).cuda()
    valid = gt > 0
    
    def calc_epe(pred, gt_val, mask):
        return torch.abs(pred - gt_val)[mask].mean().item()

    epe_fast = calc_epe(disp_fast, gt, valid)
    epe_prec = calc_epe(disp_prec_LR, gt, valid)

    print("-" * 30)
    print(f"EPE Fast Mode   : {epe_fast:.4f} px")
    print(f"EPE Precise Mode: {epe_prec:.4f} px")
    print("-" * 30)
except FileNotFoundError:
    print("GT Datei nicht gefunden. Überspringe EPE Check.")


In [ ]:
import torch
from your_model_file import StereoNetLite_GrabberCore

# 1. Load trained model
model = StereoNetLite_GrabberCore(max_disp=192, num_groups=4, input_size=(480, 640))
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# 2. Create dummy inputs
dummy_left = torch.randn(1, 1, 480, 640)
dummy_right = torch.randn(1, 1, 480, 640)

# 3. Test forward pass
with torch.no_grad():
    output = model(dummy_left, dummy_right, training=False)
    print(f"Output shape: {output.shape}")  # Should be [1, 1, 480, 640]

# 4. Export to ONNX
torch.onnx.export(
    model,
    (dummy_left, dummy_right),
    "stereo_hailo.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['left_image', 'right_image'],
    output_names=['disparity'],
    dynamic_axes={
        'left_image': {0: 'batch_size'},
        'right_image': {0: 'batch_size'},
        'disparity': {0: 'batch_size'}
    }
)

print("✅ ONNX export successful: stereo_hailo.onnx")

# 5. Verify ONNX
import onnx
onnx_model = onnx.load("stereo_hailo.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX model is valid")

# 6. Test ONNX inference
import onnxruntime as ort
session = ort.InferenceSession("stereo_hailo.onnx")
onnx_output = session.run(
    None,
    {'left_image': dummy_left.numpy(), 'right_image': dummy_right.numpy()}
)
print(f"✅ ONNX inference successful, output shape: {onnx_output[0].shape}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Deine vorhandenen Hilfsfunktionen (PFM & Bilder) ---
def read_pfm(file):
    with open(file, "rb") as f:
        header = f.readline().decode('utf-8').rstrip()
        if header != 'Pf': raise Exception('Keine PFM Pf-Datei.')
        dims = f.readline().decode('utf-8').split()
        width, height = int(dims[0]), int(dims[1])
        scale = float(f.readline().decode('utf-8').rstrip())
        endian = '<' if scale < 0 else '>'
        data = np.fromfile(f, endian + 'f')
        data = np.reshape(data, (height, width))
        data = np.flipud(data)
        data[data == np.inf] = 0
        return data.copy()

# --- 2. Die Analyse-Funktion ---
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def analyze_checkpoint_pil(checkpoint_path, left_path, right_path, gt_path, row_y=120):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_size = (640, 480) # (W, H)
    
    # 1. Modell laden (weights_only=True für Sicherheit)
    model = StereoFusionAllMode().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()

    # 2. Bilder laden & Resizen (Dein Code)
    left_img_pil = Image.open(left_path).convert('L').resize(target_size, Image.BILINEAR)
    right_img_pil = Image.open(right_path).convert('L').resize(target_size, Image.BILINEAR)
    
    # In Tensor umwandeln [1, 1, 480, 640]
    img_l = torch.from_numpy(np.array(left_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)
    img_r = torch.from_numpy(np.array(right_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)

    # 3. Ground Truth laden & Resizen
    gt_orig = read_pfm(gt_path) # Nutzt deine Funktion
    orig_h, orig_w = gt_orig.shape
    
    # WICHTIG: Wenn wir das Bild verkleinern, müssen wir die Disparitätswerte skalieren!
    scale_factor = target_size[0] / orig_w
    gt_rescaled = F.interpolate(torch.from_numpy(gt_orig).unsqueeze(0).unsqueeze(0), 
                                size=(target_size[1], target_size[0]), 
                                mode='nearest').squeeze().numpy()
    gt_rescaled = gt_rescaled * scale_factor # Werte an neue Auflösung anpassen

    # 4. Inferenz
    with torch.no_grad():
        outputs = model(img_l, img_r, mode="precise")
        pred_lr = outputs["LR"][0].cpu().squeeze().numpy()

    # 5. Plotten
    plt.figure(figsize=(15, 6))
    x = np.arange(target_size[0])
    
    plt.plot(x, gt_rescaled[row_y, :], color='black', label='GT (PFM, skaliert)', linewidth=2)
    plt.plot(x, pred_lr[row_y, :], color='red', label='Prediction LR', alpha=0.8)
    
    plt.fill_between(x, gt_rescaled[row_y, :], pred_lr[row_y, :], 
                     where=(np.abs(pred_lr[row_y, :] - gt_rescaled[row_y, :]) > 3),
                     color='red', alpha=0.1, label='Fehler > 3px')

    plt.title(f"Profil-Check Zeile {row_y} (Skalierung: {orig_w} -> {target_size[0]})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Aufruf

# ================================================================
# --- 3. DER FUNKTIONSAUFRUF (HIER PASSIERT ES) ---
# ================================================================

if __name__ == "__main__":
    # Pfade anpassen!
    MY_CHECKPOINT = "StereoFusion_5.pth"
    TEST_L = "/home/slarc/datasets/sceneflow/left/0000052.png"
    TEST_R = "/home/slarc/datasets/sceneflow/right/0000052.png"
    TEST_GT = "/home/slarc/datasets/sceneflow/disp/0000052.pfm"

    # Aufruf für die Problem-Zone (Stuhlbein-Echo oben links)
    analyze_checkpoint(
        checkpoint_path=MY_CHECKPOINT,
        left_path=TEST_L,
        right_path=TEST_R,
        gt_path=TEST_GT,
        row_y=120  # Wähle die Zeile, in der das Stuhlbein im Bild sitzt
    )

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files

# 1. Function Call
# Replace 'path_to_your_file.pfm' with your actual file path
file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/right/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()
except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/left/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()

except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files